In [30]:
%pip install -q networkx python-louvain plotly pandas pyarrow numpy

You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [31]:
INPUT_TICKER = "NFLX"  # ← Set this to match the ticker used in notebook_01

In [32]:
try:
    import pandas as pd
    import numpy as np
    import os
    from pathlib import Path

    _required = {
        "data/lead_lag_matrix.parquet": "notebook_03",
        "data/granger_results.parquet": "notebook_03",
        "data/sentiment_daily.parquet": "notebook_02",
    }
    for fpath, src in _required.items():
        if not Path(fpath).exists():
            raise FileNotFoundError(f"{fpath} not found — run {src} first")

    # index_col is not a valid read_parquet kwarg — parquet restores the ticker index automatically
    df_ll = pd.read_parquet("data/lead_lag_matrix.parquet")
    df_granger = pd.read_parquet("data/granger_results.parquet")
    df_sentiment = pd.read_parquet("data/sentiment_daily.parquet")

    print(f"lead_lag_matrix:   {df_ll.shape}  (rows=tickers, cols=tickers)")
    print(f"granger_results:   {df_granger.shape}  columns: {list(df_granger.columns)}")
    print(f"sentiment_daily:   {df_sentiment.shape}  columns: {list(df_sentiment.columns)}")
except Exception as e:
    print(f"\n[ERROR] {type(e).__name__}: {e}")
    raise SystemExit(1) from None

lead_lag_matrix:   (10, 10)  (rows=tickers, cols=tickers)
granger_results:   (10, 10)  columns: ['ticker', 'granger_verified', 'optimal_lag', 'min_p_value', 'p_value_lag1', 'p_value_lag2', 'p_value_lag3', 'p_value_lag4', 'p_value_lag5', 'skip_reason']
sentiment_daily:   (7817, 6)  columns: ['date', 'ticker', 'sentiment_score', 'article_count', 'avg_finbert_score', 'avg_lm_score']


In [33]:
try:
    TICKERS = list(df_ll.index)

    if INPUT_TICKER not in TICKERS:
        raise ValueError(
            f"INPUT_TICKER '{INPUT_TICKER}' not found in lead-lag matrix. "
            f"Re-run notebook_01 with this ticker."
        )

    print(f"INPUT_TICKER: {INPUT_TICKER}")
    print(f"Ticker universe: {TICKERS}")
except Exception as e:
    print(f"\n[ERROR] {type(e).__name__}: {e}")
    raise SystemExit(1) from None

INPUT_TICKER: NFLX
Ticker universe: ['AAPL', 'AMD', 'AMZN', 'GOOGL', 'INTC', 'META', 'MSFT', 'NFLX', 'NVDA', 'TSLA']


In [34]:
try:
    import networkx as nx

    G = nx.Graph()
    G.add_nodes_from(TICKERS)

    for i in TICKERS:
        for j in TICKERS:
            if i >= j:  # upper triangle only — avoids duplicate undirected edges
                continue
            w = abs(df_ll.loc[i, j])
            if w > 0.05:
                G.add_edge(i, j, weight=w)

    print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges (|L| > 0.05 threshold)")

    if G.number_of_edges() == 0:
        print("WARNING: no edges meet the 0.05 threshold — all tickers will be set to follower")
except Exception as e:
    print(f"\n[ERROR] {type(e).__name__}: {e}")
    raise SystemExit(1) from None

Graph: 10 nodes, 40 edges (|L| > 0.05 threshold)


In [35]:
try:
    import community as community_louvain

    if G.number_of_edges() == 0:
        partition = {t: 0 for t in TICKERS}
    else:
        partition = community_louvain.best_partition(G)

    n_partitions = len(set(partition.values()))
    print(f"Louvain detected {n_partitions} partition(s)")
    print(f"Partition assignments: {partition}")

    if n_partitions == 1:
        print("WARNING: only one partition detected — clustering inconclusive, all tickers set to follower")
except Exception as e:
    print(f"\n[ERROR] {type(e).__name__}: {e}")
    raise SystemExit(1) from None

Louvain detected 2 partition(s)
Partition assignments: {'AAPL': 0, 'AMD': 1, 'AMZN': 0, 'GOOGL': 1, 'INTC': 1, 'META': 1, 'MSFT': 1, 'NFLX': 0, 'NVDA': 0, 'TSLA': 0}


In [36]:
try:
    if n_partitions <= 1:
        cluster_map = {t: "follower" for t in TICKERS}
        print("All tickers assigned to follower (single/zero partition)")
    else:
        part_ids = set(partition.values())
        outflow_records = []

        for pid in part_ids:
            members = [t for t, p in partition.items() if p == pid]
            non_members = [t for t in TICKERS if t not in members]
            # net outflow uses SIGNED L values (not absolute weights)
            net_outflow = sum(df_ll.loc[i, j] for i in members for j in non_members)
            outflow_records.append({"pid": pid, "net_outflow": net_outflow, "members": members})

        # highest net outflow → leaders; all others → followers
        leader_record = max(outflow_records, key=lambda r: r["net_outflow"])
        leader_pid = leader_record["pid"]

        cluster_map = {
            t: ("leader" if partition[t] == leader_pid else "follower")
            for t in TICKERS
        }

        n_leaders = sum(1 for v in cluster_map.values() if v == "leader")
        n_followers = len(TICKERS) - n_leaders
        print(f"Leaders ({n_leaders}): {[t for t, r in cluster_map.items() if r == 'leader']}")
        print(f"Followers ({n_followers}): {[t for t, r in cluster_map.items() if r == 'follower']}")
        print(f"Leader partition (pid={leader_pid}) had net outflow = {leader_record['net_outflow']:.4f}")
        if n_partitions > 2:
            print(f"Note: {n_partitions} partitions returned by Louvain; all non-leader partitions merged into follower group")
except Exception as e:
    print(f"\n[ERROR] {type(e).__name__}: {e}")
    raise SystemExit(1) from None

Leaders (5): ['AAPL', 'AMZN', 'NFLX', 'NVDA', 'TSLA']
Followers (5): ['AMD', 'GOOGL', 'INTC', 'META', 'MSFT']
Leader partition (pid=0) had net outflow = 1.5188


In [37]:
try:
    try:
        centrality = nx.eigenvector_centrality(G, max_iter=1000, weight="weight")
    except nx.PowerIterationFailedConvergence:
        print("WARNING: eigenvector centrality failed to converge — falling back to degree centrality")
        centrality = nx.degree_centrality(G)

    # min-max normalize to [0.0, 1.0]
    min_c = min(centrality.values())
    max_c = max(centrality.values())

    if max_c == min_c:
        centrality_norm = {t: 0.5 for t in TICKERS}
    else:
        centrality_norm = {
            t: (centrality[t] - min_c) / (max_c - min_c)
            for t in TICKERS
        }

    print("Eigenvector centrality (normalized):")
    for t in sorted(centrality_norm, key=centrality_norm.get, reverse=True):
        print(f"  {t}: {centrality_norm[t]:.4f}  ({cluster_map[t]})")
except Exception as e:
    print(f"\n[ERROR] {type(e).__name__}: {e}")
    raise SystemExit(1) from None

Eigenvector centrality (normalized):
  META: 1.0000  (follower)
  AMZN: 0.6750  (leader)
  NFLX: 0.6271  (leader)
  GOOGL: 0.6144  (follower)
  MSFT: 0.3586  (follower)
  TSLA: 0.3144  (leader)
  AMD: 0.2313  (follower)
  INTC: 0.1646  (follower)
  AAPL: 0.1416  (leader)
  NVDA: 0.0000  (leader)


In [38]:
try:
    df_clusters = pd.DataFrame([
        {"ticker": t, "cluster": cluster_map[t], "centrality_score": centrality_norm[t]}
        for t in TICKERS
    ])

    os.makedirs("data", exist_ok=True)
    df_clusters.to_parquet("data/cluster_assignments.parquet", index=False)

    print(f"Saved data/cluster_assignments.parquet — {len(df_clusters)} tickers")
    print(df_clusters.groupby("cluster").size().to_string())
except Exception as e:
    print(f"\n[ERROR] {type(e).__name__}: {e}")
    raise SystemExit(1) from None

Saved data/cluster_assignments.parquet — 10 tickers
cluster
follower    5
leader      5


In [39]:
try:
    pos = nx.spring_layout(G, seed=42)

    # isolated nodes (no edges) won't appear in pos — assign a default position
    for t in TICKERS:
        pos.setdefault(t, (0.0, 0.0))

    print(f"Spring layout computed for {len(pos)} nodes (seed=42)")
except Exception as e:
    print(f"\n[ERROR] {type(e).__name__}: {e}")
    raise SystemExit(1) from None

Spring layout computed for 10 nodes (seed=42)


In [40]:
try:
    import plotly.graph_objects as go

    def scale_to_range(val, vmin, vmax, out_min, out_max):
        if vmax == vmin:
            return out_min
        return out_min + (val - vmin) * (out_max - out_min) / (vmax - vmin)

    # --- EDGES ---
    edge_weights = [d["weight"] for *_, d in G.edges(data=True)]
    min_edge_w = min(edge_weights) if edge_weights else 0.0
    max_edge_w = max(edge_weights) if edge_weights else 1.0

    edge_traces = []
    for u, v, d in G.edges(data=True):
        thickness = scale_to_range(d["weight"], min_edge_w, max_edge_w, 1, 5)
        edge_traces.append(
            go.Scatter(
                x=[pos[u][0], pos[v][0], None],
                y=[pos[u][1], pos[v][1], None],
                mode="lines",
                line=dict(width=thickness, color="#aaaaaa"),
                hoverinfo="none",
            )
        )

    # --- NODES ---
    node_x = [pos[t][0] for t in TICKERS]
    node_y = [pos[t][1] for t in TICKERS]
    node_colors = ["steelblue" if cluster_map[t] == "leader" else "darkorange" for t in TICKERS]
    node_sizes = [scale_to_range(centrality_norm[t], 0.0, 1.0, 20, 60) for t in TICKERS]

    node_trace = go.Scatter(
        x=node_x,
        y=node_y,
        mode="markers+text",
        text=TICKERS,
        textposition="top center",
        marker=dict(
            size=node_sizes,
            color=node_colors,
            line=dict(width=1, color="#333333"),
        ),
        hovertext=[
            f"{t} ({cluster_map[t]}, centrality={centrality_norm[t]:.3f})"
            for t in TICKERS
        ],
        hoverinfo="text",
    )

    # --- FIGURE ---
    fig = go.Figure(
        data=edge_traces + [node_trace],
        layout=go.Layout(
            title=f"Lead-Lag Network — {INPUT_TICKER} highlighted",
            showlegend=False,
            hovermode="closest",
            xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            margin=dict(l=20, r=20, t=40, b=20),
        ),
    )

    print(f"Plotly figure built: {len(edge_traces)} edge trace(s) + 1 node trace")
except Exception as e:
    print(f"\n[ERROR] {type(e).__name__}: {e}")
    raise SystemExit(1) from None

Plotly figure built: 40 edge trace(s) + 1 node trace


In [41]:
try:
    os.makedirs("outputs", exist_ok=True)
    fig.write_html("outputs/network_graph.html", full_html=True)

    size_kb = Path("outputs/network_graph.html").stat().st_size / 1024
    print(f"Saved outputs/network_graph.html — {size_kb:.1f} KB")
except Exception as e:
    print(f"\n[ERROR] {type(e).__name__}: {e}")
    raise SystemExit(1) from None

Saved outputs/network_graph.html — 4745.8 KB


In [42]:
try:
    if INPUT_TICKER not in cluster_map:
        print(
            f"INPUT_TICKER '{INPUT_TICKER}' was not found in cluster_assignments. "
            f"Re-run from Cell 5 (graph construction) to rebuild cluster_map, "
            f"or verify INPUT_TICKER is in the lead-lag matrix."
        )
        raise SystemExit(1) from None

    input_role = cluster_map[INPUT_TICKER]

    if input_role == "leader":
        counterparts = [t for t in TICKERS if cluster_map[t] == "follower"]
        rows = []
        for t in counterparts:
            lag_rows = df_granger.loc[df_granger["ticker"] == t, "optimal_lag"]
            rows.append({
                "related_ticker": t,
                "lead_lag_score": df_ll.loc[INPUT_TICKER, t],
                "optimal_lag_days": int(lag_rows.values[0]) if len(lag_rows) > 0 and pd.notna(lag_rows.values[0]) else None,
            })
    else:
        counterparts = [t for t in TICKERS if cluster_map[t] == "leader"]
        rows = []
        for t in counterparts:
            lag_rows = df_granger.loc[df_granger["ticker"] == t, "optimal_lag"]
            rows.append({
                "related_ticker": t,
                "lead_lag_score": df_ll.loc[t, INPUT_TICKER],
                "optimal_lag_days": int(lag_rows.values[0]) if len(lag_rows) > 0 and pd.notna(lag_rows.values[0]) else None,
            })

    _cols = ["related_ticker", "lead_lag_score", "optimal_lag_days"]
    if not rows:
        df_suggestion = pd.DataFrame(columns=_cols)
    else:
        df_suggestion = (
            pd.DataFrame(rows)
            .sort_values("lead_lag_score", ascending=False, key=abs)
            .reset_index(drop=True)
        )

    print(f"\n=== Suggestion Table for {INPUT_TICKER} ({input_role.upper()}) ===")
    pd.set_option("display.max_rows", None)
    if len(df_suggestion) == 0:
        print(f"(no counterpart stocks — all tickers share the same cluster as {INPUT_TICKER})")
    else:
        print(df_suggestion.to_string(index=False))
except Exception as e:
    print(f"\n[ERROR] {type(e).__name__}: {e}")
    raise SystemExit(1) from None


=== Suggestion Table for NFLX (LEADER) ===
related_ticker  lead_lag_score  optimal_lag_days
          META        0.321787                 1
         GOOGL        0.312903                 3
          INTC        0.261965                 1
          MSFT       -0.188462                 1
           AMD        0.036048                 5


In [43]:
try:
    granger_row = df_granger[df_granger["ticker"] == INPUT_TICKER]
    granger_verified = bool(granger_row["granger_verified"].values[0]) if len(granger_row) > 0 else False
    opt_lag = granger_row["optimal_lag"].values[0] if len(granger_row) > 0 else None
    min_p = granger_row["min_p_value"].values[0] if len(granger_row) > 0 else None

    # collect top 2 counterpart names for the summary sentence
    top_names = list(df_suggestion["related_ticker"].iloc[:2]) if len(df_suggestion) > 0 else []
    if len(top_names) == 0:
        top_names_str = "no counterpart stocks identified"
    elif len(top_names) == 1:
        top_names_str = top_names[0]
    else:
        top_names_str = f"{top_names[0]} and {top_names[1]}"

    if input_role == "leader":
        role_text = f"{INPUT_TICKER} is a LEADER stock in this universe."
        direction_text = f"It leads follower stocks such as {top_names_str}."
    else:
        role_text = f"{INPUT_TICKER} is a FOLLOWER stock in this universe."
        direction_text = f"It is influenced by leader stocks such as {top_names_str}."

    if granger_verified:
        granger_text = (
            f"Granger causality analysis CONFIRMS that sentiment for {INPUT_TICKER} "
            f"statistically predicts its price movement "
            f"(p={min_p:.4f}, optimal lag={int(opt_lag)} trading day(s))."
        )
    else:
        p_str = f"{min_p:.4f}" if min_p is not None and pd.notna(min_p) else "N/A"
        granger_text = (
            f"Granger causality analysis did NOT find statistically significant "
            f"sentiment-to-price predictability for {INPUT_TICKER} "
            f"(min p={p_str}). Treat these signals with caution."
        )

    print(f"\n=== Plain-Language Summary ===")
    print(f"{role_text} {direction_text}")
    print(f"{granger_text}")
except Exception as e:
    print(f"\n[ERROR] {type(e).__name__}: {e}")
    raise SystemExit(1) from None


=== Plain-Language Summary ===
NFLX is a LEADER stock in this universe. It leads follower stocks such as META and GOOGL.
Granger causality analysis CONFIRMS that sentiment for NFLX statistically predicts its price movement (p=0.0003, optimal lag=1 trading day(s)).


## EDA — Network & Clustering Exploratory Plots

In [ ]:

# EDA 12 — Eigenvector Centrality Ranking Bar Chart
import matplotlib.pyplot as plt
import pandas as pd

df_clust = pd.read_parquet("data/cluster_assignments.parquet")
df_clust = df_clust.sort_values("centrality_score", ascending=True)

colors_role = ['steelblue' if r == 'leader' else 'darkorange'
               for r in df_clust["cluster"]]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(df_clust["ticker"], df_clust["centrality_score"],
               color=colors_role, edgecolor='white', height=0.6)

for bar, (_, row) in zip(bars, df_clust.iterrows()):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f"{row['centrality_score']:.3f}  ({row['cluster']})",
            va='center', fontsize=10)

import matplotlib.patches as mpatches
ax.legend(handles=[
    mpatches.Patch(color='steelblue',   label='Leader'),
    mpatches.Patch(color='darkorange',  label='Follower'),
], fontsize=10)

ax.set_title("Eigenvector Centrality Ranking by Ticker\n(min-max normalised, max_iter=1000)",
             fontsize=13, fontweight='bold')
ax.set_xlabel("Normalised Centrality Score", fontsize=12)
ax.set_xlim(0, 1.25)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig("outputs/eda_12_centrality_ranking.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved outputs/eda_12_centrality_ranking.png")


In [ ]:

# EDA 13 — Net Outflow per Ticker (signed lead-lag influence)
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

df_ll_eda = pd.read_parquet("data/lead_lag_matrix.parquet")
df_clust  = pd.read_parquet("data/cluster_assignments.parquet")
role_map  = dict(zip(df_clust["ticker"], df_clust["cluster"]))
tickers   = list(df_ll_eda.index)

net_outflows = {}
for t in tickers:
    others = [o for o in tickers if o != t]
    net_outflows[t] = sum(df_ll_eda.loc[t, o] for o in others)

df_outflow = pd.Series(net_outflows).sort_values(ascending=False)
colors_out = ['steelblue' if role_map.get(t) == 'leader' else 'darkorange'
              for t in df_outflow.index]

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(df_outflow.index, df_outflow.values, color=colors_out, edgecolor='white')
ax.axhline(0, color='black', linewidth=0.8)
for bar, val in zip(bars, df_outflow.values):
    ax.text(bar.get_x() + bar.get_width()/2,
            val + (0.02 if val >= 0 else -0.05),
            f"{val:+.2f}", ha='center', va='bottom' if val >= 0 else 'top', fontsize=9)

import matplotlib.patches as mpatches
ax.legend(handles=[
    mpatches.Patch(color='steelblue',  label='Leader'),
    mpatches.Patch(color='darkorange', label='Follower'),
], fontsize=10)
ax.set_title("Net Outflow per Ticker  [Σ L(i,j) over all j ≠ i]\n"
             "Positive = net leader;  Negative = net follower",
             fontsize=13, fontweight='bold')
ax.set_xlabel("Ticker", fontsize=12)
ax.set_ylabel("Net Outflow Score", fontsize=12)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig("outputs/eda_13_net_outflow.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved outputs/eda_13_net_outflow.png")
